# Chapter 18c — Muon on CUDA/C++: Minimal and Optimal

> Course: **llm.c — Zero to Hero**, companion to Chapter 18 (GPU AdamW).
> Builds on: **Chapter 18a/18b** (Muon math + numpy implementation), Chapter 13 (reductions),
> Chapter 14a (GPU matmul), Chapter 18 (the GPU optimizer step & `cudaEvent` timing).
>
> Audience: **freshman** — you've written a CUDA kernel before (grid-stride loop, `nvcc`,
> `cudaEvent` timing). We now make Muon's core operation fast on the GPU.

In 18b you saw the comment *"the ONE expensive op is `A = X Xᵀ`"*. Newton-Schulz runs that line
**5 times per matrix, every optimizer step**, across thousands of weight matrices. This chapter
implements Newton-Schulz in CUDA twice:

1. a **minimal** version — three plain matmul kernels, verified against a CPU reference;
2. an **optimal** version — the **flash-muon** trick: `X Xᵀ` is *symmetric*, so compute only the
   upper triangle and mirror it, halving the work. We benchmark the real speedup on the GPU.

### Learning objectives

- Express one Newton-Schulz step as a sequence of GPU matmul + elementwise kernels.
- Verify a GPU kernel against a CPU reference to machine precision.
- Explain why `X Xᵀ` is symmetric and how to skip the redundant lower-triangle tiles.
- Measure the speedup of the symmetric kernel with `cudaEvent`, and explain why it's ~2× (not more).


## 1. Where the time goes

One Newton-Schulz step, from 18a/18b, on an `n × n` matrix `X`:

```
A  = X X^T            # matmul  (the symmetric one)
A2 = A  A             # matmul
B  = b*A + c*A2       # elementwise
X  = a*X + B X        # matmul + elementwise
```

That's **three `n×n×n` matmuls per step**, times 5 steps. Matmul is `O(n³)`, the elementwise ops are
`O(n²)` — so the matmuls dominate completely. Of those three, the first, `A = X Xᵀ`, has special
structure we can exploit: its result is **symmetric**. Hold that thought for §3.

We'll build everything in `course/ch18c_build/`.


In [ ]:
!mkdir -p course/ch18c_build


## 2. Minimal Newton-Schulz on the GPU

Three kernels: a naive `matmul` (with an optional "transpose the second operand" flag so we can
reuse it for `X Xᵀ`), and an `axpby` for the elementwise `a*X + b*Z` combines. We run 5 steps on
the GPU and check against a straight C translation of the same math on the CPU.


In [ ]:
%%writefile course/ch18c_build/ns_minimal.cu
// Minimal Newton-Schulz orthogonalization on the GPU.
// One quintic step:  A = X X^T ;  X <- a*X + (b*A + c*A^2) @ X.
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define TILE 16
static const float A_NS = 3.4445f, B_NS = -4.7750f, C_NS = 2.0315f;

// naive matmul:  C = A @ (transB ? B^T : B), all n x n
__global__ void matmul(const float* A, const float* B, float* C, int n, int transB) {
    int r = blockIdx.y * blockDim.y + threadIdx.y;
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    if (r < n && c < n) {
        float s = 0.0f;
        for (int k = 0; k < n; k++) {
            float b = transB ? B[c * n + k] : B[k * n + c];
            s += A[r * n + k] * b;
        }
        C[r * n + c] = s;
    }
}
// Y = a*X + b*Z   (elementwise over the whole n*n matrix)
__global__ void axpby(float* Y, float a, const float* X, float b, const float* Z, int nn) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < nn) Y[i] = a * X[i] + b * Z[i];
}

// CPU reference: the exact same math, plain triple loops.
void ns_cpu(float* X, int n, int steps) {
    float *A=(float*)malloc(n*n*4),*A2=(float*)malloc(n*n*4),*B=(float*)malloc(n*n*4),*BX=(float*)malloc(n*n*4);
    for (int s=0;s<steps;s++){
        for(int i=0;i<n;i++)for(int j=0;j<n;j++){float v=0;for(int k=0;k<n;k++)v+=X[i*n+k]*X[j*n+k];A[i*n+j]=v;}
        for(int i=0;i<n;i++)for(int j=0;j<n;j++){float v=0;for(int k=0;k<n;k++)v+=A[i*n+k]*A[k*n+j];A2[i*n+j]=v;}
        for(int i=0;i<n*n;i++)B[i]=B_NS*A[i]+C_NS*A2[i];
        for(int i=0;i<n;i++)for(int j=0;j<n;j++){float v=0;for(int k=0;k<n;k++)v+=B[i*n+k]*X[k*n+j];BX[i*n+j]=v;}
        for(int i=0;i<n*n;i++)X[i]=A_NS*X[i]+BX[i];
    }
    free(A);free(A2);free(B);free(BX);
}

int main(void) {
    int n = 256, steps = 5;
    size_t sz = (size_t)n*n*4;
    float *hX=(float*)malloc(sz), *hCPU=(float*)malloc(sz), *hGPU=(float*)malloc(sz);
    srand(1);
    double nrm = 0;
    for (int i=0;i<n*n;i++){ hX[i]=((rand()%1000)/1000.0f-0.5f); nrm+=(double)hX[i]*hX[i]; }
    nrm = sqrt(nrm) + 1e-7;                       // Frobenius normalize -> singular values <= 1
    for (int i=0;i<n*n;i++) hX[i] /= (float)nrm;
    memcpy(hCPU, hX, sz); ns_cpu(hCPU, n, steps);

    float *X,*A,*A2,*B,*BX;
    cudaMalloc(&X,sz);cudaMalloc(&A,sz);cudaMalloc(&A2,sz);cudaMalloc(&B,sz);cudaMalloc(&BX,sz);
    cudaMemcpy(X, hX, sz, cudaMemcpyHostToDevice);
    dim3 blk(TILE,TILE), grd((n+TILE-1)/TILE,(n+TILE-1)/TILE);
    int t1 = (n*n + 255) / 256;
    for (int s = 0; s < steps; s++) {
        matmul<<<grd,blk>>>(X, X, A, n, 1);       // A  = X X^T
        matmul<<<grd,blk>>>(A, A, A2, n, 0);      // A2 = A A
        axpby<<<t1,256>>>(B, B_NS, A, C_NS, A2, n*n);    // B = b*A + c*A2
        matmul<<<grd,blk>>>(B, X, BX, n, 0);      // BX = B X
        axpby<<<t1,256>>>(X, A_NS, X, 1.0f, BX, n*n);    // X = a*X + BX
    }
    cudaDeviceSynchronize();
    cudaMemcpy(hGPU, X, sz, cudaMemcpyDeviceToHost);
    float md = 0; for (int i=0;i<n*n;i++) md = fmaxf(md, fabsf(hGPU[i]-hCPU[i]));
    printf("Newton-Schulz (minimal GPU) vs CPU reference: max abs diff = %.3e\n", md);
    printf("%s\n", md < 1e-3 ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18c_build/ns_minimal course/ch18c_build/ns_minimal.cu && ./course/ch18c_build/ns_minimal


The GPU result matches the CPU triple-loop reference to ~`1e-6` — float32 rounding from the
different summation order, nothing more. **You just orthogonalized a 256×256 matrix on the GPU with
three matmul kernels.** Now let's make the expensive `X Xᵀ` step ~2× cheaper.


## 3. The optimal trick — `X Xᵀ` is symmetric

For any matrix `X`, the product `A = X Xᵀ` satisfies `Aᵀ = (X Xᵀ)ᵀ = X Xᵀ = A`: it is
**symmetric**. So the entry `A[r][c]` equals `A[c][r]` — we are computing the **lower triangle
twice for nothing**.

### See it on real numbers first

Forget the GPU and tiles for a moment. The rule for `A = X Xᵀ` is just:
**`A[i][j] = dot product of row i and row j of `X``**. Take a tiny `3×3`:

```
        col0 col1 col2
row0 →  [ 1    2    3 ]
row1 →  [ 4    5    6 ]
row2 →  [ 7    8    9 ]
```

Fill in `A` by hand, one dot product per entry:

```
A[0][0] = row0·row0 = 1·1 + 2·2 + 3·3 = 14
A[0][1] = row0·row1 = 1·4 + 2·5 + 3·6 = 32
A[0][2] = row0·row2 = 1·7 + 2·8 + 3·9 = 50
A[1][1] = row1·row1 = 4·4 + 5·5 + 6·6 = 77
A[1][2] = row1·row2 = 4·7 + 5·8 + 6·9 = 122
A[2][2] = row2·row2 = 7·7 + 8·8 + 9·9 = 194

A[1][0] = row1·row0 = 4·1 + 5·2 + 6·3 = 32    ← identical to A[0][1]
A[2][0] = row2·row0 = 50                       ← identical to A[0][2]
A[2][1] = row2·row1 = 122                      ← identical to A[1][2]
```

```
        col0 col1 col2
row0 →  [ 14   32   50 ]
row1 →  [ 32   77  122 ]
row2 →  [ 50  122  194 ]
```

`A[0][1]` and `A[1][0]` are **both 32** — and not by luck: `row0·row1` runs the *same multiply-adds*
as `row1·row0` (a dot product ignores order). Fold the matrix along its diagonal (`14, 77, 194`) and
the two halves match. So you only need the **6 upper-triangle values**; the 3 lower-left ones are
free copies:

```
compute upper triangle ──┐                [ 14   32   50 ]
[ 14   32   50 ]         │   mirror        [ 32   77  122 ]
[      77  122 ]         ├───────────────► [ 50  122  194 ]
[          194 ]         │   A[j][i]=A[i][j]
                       (copy 32, 50, 122 across the diagonal)
```

That "compute the upper triangle once, copy each value to its mirror slot" **is the entire
optimization**. Everything below is just doing it efficiently on a GPU. The flash-muon kernel's own
words:

> *"We only calculate upper triangular parts of the result, and then transpose and copy each result
> tile to the corresponding lower triangular parts"* — saving ~half the GEMM work.


### Scale the idea up to tiles

A GPU dislikes skipping *individual* elements — it launches work in fixed `16×16` **tiles**. So we do
the exact same trick one zoom-level up: instead of "skip the lower-left **elements**," we **skip the
lower-left tiles**. We launch a block **only** for tiles on or above the diagonal; each off-diagonal
tile is computed once and written to both `A[r][c]` and `A[c][r]`. (The per-block bookkeeping that
turns a flat `blockIdx.x` into a tile coordinate is just plumbing — the idea is the picture above.)

```mermaid
flowchart LR
  subgraph FULL["naive: compute ALL tiles"]
    f["T x T tiles"]
  end
  subgraph SYM["symmetric: compute UPPER tiles only"]
    u["T*(T+1)/2 tiles"]
    mir["mirror each to its lower twin"]
    u --> mir
  end
  FULL -- "~half the tiles are redundant" --> SYM
```

For `T` tiles per side, the naive kernel launches `T²` tile-blocks; the symmetric one launches
`T·(T+1)/2` — about **half**. That is the entire saving, and it's exact (no approximation).


In [ ]:
%%writefile course/ch18c_build/ns_optimal.cu
// Optimal Newton-Schulz: exploit that A = X X^T is SYMMETRIC (the flash-muon trick).
// Launch ONLY the upper-triangular output tiles and mirror each into the lower triangle.
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define TILE 16
static const float A_NS = 3.4445f, B_NS = -4.7750f, C_NS = 2.0315f;

__global__ void matmul(const float* A, const float* B, float* C, int n, int transB) {
    int r = blockIdx.y * blockDim.y + threadIdx.y;
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    if (r < n && c < n) {
        float s = 0.0f;
        for (int k = 0; k < n; k++) { float b = transB ? B[c*n+k] : B[k*n+c]; s += A[r*n+k]*b; }
        C[r*n+c] = s;
    }
}
// symmetric C = X X^T. 1D grid over the T*(T+1)/2 upper-triangular tiles.
__global__ void matmul_sym(const float* X, float* C, int n, int T) {
    int m = blockIdx.x;                          // which upper-tri tile (linear index)
    int br = 0; while (m >= T - br) { m -= (T - br); br++; }   // decode -> tile row
    int bc = br + m;                             // tile col, guaranteed bc >= br
    int r = br*TILE + threadIdx.y;
    int c = bc*TILE + threadIdx.x;
    if (r < n && c < n) {
        float s = 0.0f;
        for (int k = 0; k < n; k++) s += X[r*n+k] * X[c*n+k];
        C[r*n+c] = s;
        if (br != bc) C[c*n+r] = s;              // mirror off-diagonal tiles into the lower triangle
    }
}
__global__ void axpby(float* Y, float a, const float* X, float b, const float* Z, int nn) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < nn) Y[i] = a*X[i] + b*Z[i];
}

int main(void) {
    int n = 1024, steps = 5;
    int T = (n+TILE-1)/TILE, ntri = T*(T+1)/2;
    size_t sz = (size_t)n*n*4;
    float *hX=(float*)malloc(sz);
    srand(1);
    double nrm=0; for(int i=0;i<n*n;i++){hX[i]=((rand()%1000)/1000.0f-0.5f);nrm+=(double)hX[i]*hX[i];}
    nrm=sqrt(nrm)+1e-7; for(int i=0;i<n*n;i++)hX[i]/=(float)nrm;

    float *X,*A,*A2,*B,*BX; cudaMalloc(&X,sz);cudaMalloc(&A,sz);cudaMalloc(&A2,sz);cudaMalloc(&B,sz);cudaMalloc(&BX,sz);
    cudaMemcpy(X,hX,sz,cudaMemcpyHostToDevice);
    dim3 blk(TILE,TILE), grd((n+TILE-1)/TILE,(n+TILE-1)/TILE);
    int t1=(n*n+255)/256;

    // 1) correctness: the symmetric kernel must EXACTLY match the full matmul for X X^T
    matmul<<<grd,blk>>>(X, X, A, n, 1);
    matmul_sym<<<ntri,blk>>>(X, A2, n, T);
    cudaDeviceSynchronize();
    float *hA=(float*)malloc(sz), *hAs=(float*)malloc(sz);
    cudaMemcpy(hA,A,sz,cudaMemcpyDeviceToHost); cudaMemcpy(hAs,A2,sz,cudaMemcpyDeviceToHost);
    float md=0; for(int i=0;i<n*n;i++) md=fmaxf(md,fabsf(hA[i]-hAs[i]));
    printf("symmetric X X^T vs full matmul: max abs diff = %.3e  %s\n", md, md<1e-4?"PASS":"FAIL");

    // 2) benchmark the X X^T step: full matmul vs symmetric
    cudaEvent_t e0,e1; cudaEventCreate(&e0); cudaEventCreate(&e1); int reps=100;
    cudaEventRecord(e0); for(int r=0;r<reps;r++) matmul<<<grd,blk>>>(X,X,A,n,1); cudaEventRecord(e1); cudaEventSynchronize(e1);
    float tf; cudaEventElapsedTime(&tf,e0,e1);
    cudaEventRecord(e0); for(int r=0;r<reps;r++) matmul_sym<<<ntri,blk>>>(X,A,n,T); cudaEventRecord(e1); cudaEventSynchronize(e1);
    float ts; cudaEventElapsedTime(&ts,e0,e1);
    printf("X X^T  full matmul : %.3f ms/call\n", tf/reps);
    printf("X X^T  symmetric   : %.3f ms/call   (%.2fx faster)\n", ts/reps, tf/ts);

    // 3) a full Newton-Schulz using the fast symmetric kernel for the X X^T step
    cudaMemcpy(X,hX,sz,cudaMemcpyHostToDevice);
    for (int s=0;s<steps;s++){
        matmul_sym<<<ntri,blk>>>(X, A, n, T);    // A = X X^T   (fast symmetric path)
        matmul<<<grd,blk>>>(A, A, A2, n, 0);     // A2 = A A
        axpby<<<t1,256>>>(B, B_NS, A, C_NS, A2, n*n);
        matmul<<<grd,blk>>>(B, X, BX, n, 0);     // BX = B X
        axpby<<<t1,256>>>(X, A_NS, X, 1.0f, BX, n*n);
    }
    cudaDeviceSynchronize();
    float *hGPU=(float*)malloc(sz); cudaMemcpy(hGPU,X,sz,cudaMemcpyDeviceToHost);
    float ss=0; for(int i=0;i<n*n;i++) ss+=hGPU[i]*hGPU[i];
    printf("full NS (symmetric path) ||X||_F = %.2f   (orthogonal 1024x1024 -> sqrt(1024)=32.0; close => singular values near 1)\n", sqrtf(ss));
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18c_build/ns_optimal course/ch18c_build/ns_optimal.cu && ./course/ch18c_build/ns_optimal


Three results to read:

1. **Exact match** (`diff = 0`): the symmetric kernel is not an approximation — it computes the same
   numbers, just skipping redundant tiles. Correctness first, always.
2. **~2× faster** on the `X Xᵀ` step: about what the tile count predicts (`T²` → `T(T+1)/2`).
3. The final `‖X‖_F ≈ 29` for a `1024×1024` matrix, vs `√1024 = 32` for a perfectly orthogonal one —
   confirming the singular values landed **near** 1 (the speed-tuned band from 18a), on the GPU.

**Why ~2× and not more?** We only sped up *one* of the three matmuls per step, and our naive kernel
is memory-/launch-bound rather than compute-bound. A production kernel (flash-muon) also uses shared
memory tiling and fuses the polynomial, pushing further — but the symmetric saving is the single
biggest, cleanest structural win, and it's exact.


## 4. Translation Bridge

| This chapter (teaching CUDA) | Production |
|---|---|
| `matmul_sym` (upper-triangle tiles, mirror) | `flash-muon`'s `matmul_transpose(x)` CUDA kernel (`nil0x9/flash-muon`) |
| 5-step loop calling the kernels | `flash-muon`'s `fast_newtonschulz(x, steps)` |
| numpy `newton_schulz` (Ch 18b) | `KellerJordan/Muon` `zeropower_via_newtonschulz5` (bf16, PyTorch) |
| FP32 throughout | bf16 matmuls on tensor cores; the coefficients are tuned to stay stable in 16-bit |

`flash-muon` reports the `X Xᵀ` kernel cutting Newton-Schulz time by ~1.5–2× across A100 / H800 /
4090 at large dimensions — the same structural win you just measured, with production-grade tiling.


## 5. Exercises

### Exercise 1 — count the saved tiles (no code)

For `n = 1024` and `TILE = 16`, there are `T = 64` tiles per side. How many tile-blocks does the
**full** matmul launch, and how many does the **symmetric** kernel launch? What's the ratio?
**Predict** before checking with the formula.


<details>
<summary>▶ Show solution</summary>

```
full:       T*T            = 64 * 64        = 4096 tile-blocks
symmetric:  T*(T+1)/2      = 64 * 65 / 2    = 2080 tile-blocks
ratio:      4096 / 2080    ~= 1.97x  fewer blocks
```

The diagonal tiles (`br == bc`, there are `T` of them) are still computed in full, which is why the
ratio is slightly under 2× — `T(T+1)/2` not `T²/2`. As `n` grows the diagonal's share shrinks and the
ratio approaches exactly 2×.
</details>


### Exercise 2 — implement the final `axpby` update yourself

The TODO kernel below is supposed to compute `Y = a*X + b*Z` (the elementwise combine used twice per
Newton-Schulz step) but the body is missing, so the program **FAILs** its self-check. Fill it in,
then run the **Solution** cell to confirm it PASSes.


In [ ]:
%%writefile course/ch18c_build/exercise2.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

// TODO: compute Y[i] = a*X[i] + b*Z[i] for i < nn
__global__ void axpby(float* Y, float a, const float* X, float b, const float* Z, int nn) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    // TODO: guard i < nn, then Y[i] = a*X[i] + b*Z[i];
}

int main(void) {
    int nn = 4096; size_t sz = nn*4;
    float *hX=(float*)malloc(sz), *hZ=(float*)malloc(sz), *hY=(float*)malloc(sz);
    for (int i=0;i<nn;i++){ hX[i]=1.0f; hZ[i]=2.0f; }
    float *X,*Z,*Y; cudaMalloc(&X,sz);cudaMalloc(&Z,sz);cudaMalloc(&Y,sz);
    cudaMemcpy(X,hX,sz,cudaMemcpyHostToDevice); cudaMemcpy(Z,hZ,sz,cudaMemcpyHostToDevice);
    cudaMemset(Y, 0, sz);
    axpby<<<(nn+255)/256,256>>>(Y, 3.4445f, X, 1.0f, Z, nn);   // expect 3.4445*1 + 1*2 = 5.4445
    cudaMemcpy(hY,Y,sz,cudaMemcpyDeviceToHost);
    float md=0; for(int i=0;i<nn;i++) md=fmaxf(md, fabsf(hY[i]-5.4445f));
    printf("max abs error vs 5.4445 = %.3e\n", md);
    printf("%s\n", md < 1e-4 ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18c_build/exercise2 course/ch18c_build/exercise2.cu && ./course/ch18c_build/exercise2


### Solution

In [ ]:
%%writefile course/ch18c_build/exercise2_sol.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

__global__ void axpby(float* Y, float a, const float* X, float b, const float* Z, int nn) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < nn) Y[i] = a * X[i] + b * Z[i];
}

int main(void) {
    int nn = 4096; size_t sz = nn*4;
    float *hX=(float*)malloc(sz), *hZ=(float*)malloc(sz), *hY=(float*)malloc(sz);
    for (int i=0;i<nn;i++){ hX[i]=1.0f; hZ[i]=2.0f; }
    float *X,*Z,*Y; cudaMalloc(&X,sz);cudaMalloc(&Z,sz);cudaMalloc(&Y,sz);
    cudaMemcpy(X,hX,sz,cudaMemcpyHostToDevice); cudaMemcpy(Z,hZ,sz,cudaMemcpyHostToDevice);
    cudaMemset(Y, 0, sz);
    axpby<<<(nn+255)/256,256>>>(Y, 3.4445f, X, 1.0f, Z, nn);
    cudaMemcpy(hY,Y,sz,cudaMemcpyDeviceToHost);
    float md=0; for(int i=0;i<nn;i++) md=fmaxf(md, fabsf(hY[i]-5.4445f));
    printf("max abs error vs 5.4445 = %.3e\n", md);
    printf("%s\n", md < 1e-4 ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch18c_build/exercise2_sol course/ch18c_build/exercise2_sol.cu && ./course/ch18c_build/exercise2_sol


### Exercise 3 — when does the symmetric trick NOT help? (no code)

Our benchmark used `n = 1024`. **Predict:** for a tiny matrix like `n = 64` (`T = 4` tiles), would
you still see ~2×, or less? Why?


<details>
<summary>▶ Show solution</summary>

Less. Two reasons:

1. **Fixed overhead dominates.** Kernel launch latency and the `while`-loop tile decode are ~constant;
   for a tiny matrix the actual matmul work is so small that this overhead is a big fraction of the
   runtime, so halving the matmul work barely moves the total.
2. **The diagonal's share is large.** With `T = 4`, the symmetric kernel launches `4*5/2 = 10` tiles
   vs `16` full — only `1.6×` fewer, and `4` of those `10` are full-work diagonal tiles.

The trick pays off at the **large** matrix sizes real LLMs use (model widths of 1k–16k), exactly
where Muon spends its time — which is why flash-muon reports its wins at `dim = 4096–8192`.
</details>


## Further Reading

**Source of truth**

- [`nil0x9/flash-muon`](https://github.com/nil0x9/flash-muon) — the CUDA `matmul_transpose` / `fast_newtonschulz` kernels: compute the upper triangle of `X Xᵀ` and mirror it, the exact trick in §3.
- [`KellerJordan/Muon`](https://github.com/KellerJordan/Muon) — the reference `zeropower_via_newtonschulz5` whose math we implemented on the GPU.

**Going deeper**

- [_Faster Parallel Reductions on Kepler_](https://developer.nvidia.com/blog/faster-parallel-reductions-kepler/) — the reduction pattern used to compute the Frobenius norm for normalization (Chapter 18).
- Chapter 14a (this course) — tiled GPU matmul with shared memory; the next optimization to layer on top of the symmetric kernel.

**Sibling chapters**

- **Chapter 18a / 18b** — the math and the numpy implementation this chapter accelerates.


## Recap

- One Newton-Schulz step is **three `n³` matmuls**; the matmuls dominate, run 5× per matrix per step.
- **Minimal GPU version**: naive `matmul` (+ a transpose flag for `X Xᵀ`) and `axpby`; matched the CPU reference to ~`1e-6`.
- `A = X Xᵀ` is **symmetric** (`Aᵀ = A`), so the lower triangle is redundant. The **flash-muon trick** launches only the `T(T+1)/2` upper-triangular tiles and mirrors them.
- Measured **~2× faster** on the `X Xᵀ` step at `n = 1024`, with an **exact** match — a structural win, not an approximation.
- Production (`flash-muon`) adds bf16 tensor-core matmuls and shared-memory tiling on top of this same idea.

### What's next

You've now seen Muon end to end: the **math** (18a), a working **implementation** (18b), and a
**fast CUDA** core (18c). Next in the main track, **Chapter 19 — The Full Training Loop on GPU** —
where optimizers like AdamW and Muon plug into the real `train_gpt2.cu` step.
